# Notebook 05b — Reentrenamiento con KITTI + Ilusiones (v2)

**Proyecto:** Modelo Cognitivo Artificial para la Replicación de la Actividad Neurofisiológica de Percepción de Profundidad  
**Autor:** Jesús Goenaga Peña

---

## Arquitectura rediseñada: ROI Pooling comparativo

El modelo recibe la imagen completa y las coordenadas de dos regiones (A y B).  
Extrae un mapa de features global mediante la arquitectura bioinspirada (NGL→V1→V2→V3→V4→V5/MT),  
luego aplica ROI Pooling sobre las regiones A y B por separado, y finalmente compara  
las representaciones para predecir si A está más cerca o más lejos que B.

Este diseño replica el mecanismo de atención selectiva del sistema visual biológico:  
procesamiento global de la escena primero, focalización en objetos relevantes después.

## Etiquetado unificado

- **KITTI:** disparidad media de la región (mayor disparidad = más cercano al observador)
- **Ilusiones:** varianza residual de la región (validada manualmente, 15/15 correctas)
- **Ground truth:** validado y corregido en NB08-09

## Aumentación de datos

Se generan múltiples pares A/B por imagen dividiendo en cuadrícula 3×4,  
pasando de 200 pares base a ~2000-3000 pares de entrenamiento.

---

## Celda 1 — Configuración inicial

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, time
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {device}')
if device.type == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB')

BASE_DIR        = '/content/drive/MyDrive/cognitive-depth-model'
KITTI_LEFT_DIR  = os.path.join(BASE_DIR, 'data/raw/kitti/data_scene_flow/training/image_2')
KITTI_RIGHT_DIR = os.path.join(BASE_DIR, 'data/raw/kitti/data_scene_flow/training/image_3')
KITTI_DISP_DIR  = os.path.join(BASE_DIR, 'data/raw/kitti/data_scene_flow/training/disp_occ_0')
ILLUSION_DIR    = os.path.join(BASE_DIR, 'data/raw/illusions/images')
SPLITS_DIR      = os.path.join(BASE_DIR, 'data/splits')
CHECKPOINT_DIR  = os.path.join(BASE_DIR, 'checkpoints')
RESULTS_DIR     = os.path.join(BASE_DIR, 'results')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print()
print('Verificación de rutas:')
for nombre, ruta in [
    ('KITTI izq',    KITTI_LEFT_DIR),
    ('KITTI der',    KITTI_RIGHT_DIR),
    ('KITTI disp',   KITTI_DISP_DIR),
    ('Ilusiones',    ILLUSION_DIR),
    ('Splits',       SPLITS_DIR),
]:
    print(f'  {"✓" if os.path.exists(ruta) else "✗"}  {nombre}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dispositivo: cuda
  GPU: Tesla T4
  VRAM: 14.6 GB

Verificación de rutas:
  ✓  KITTI izq
  ✓  KITTI der
  ✓  KITTI disp
  ✓  Ilusiones
  ✓  Splits


## Celda 2 — Generador de pares A/B aumentados

In [10]:
def calcular_metrica_region_kitti(ruta_disp, x, y, w, h):
    """Disparidad media de una región en KITTI."""
    disp = cv2.imread(ruta_disp, cv2.IMREAD_UNCHANGED)
    if disp is None: return None
    disp = disp.astype(np.float32) / 256.0
    region = disp[y:y+h, x:x+w]
    validos = region[region > 0]
    return float(validos.mean()) if len(validos) > 10 else None

def calcular_metrica_region_ilusion(ruta_img, x, y, w, h, reg_global):
    """Varianza residual de una región en ilusiones."""
    img = cv2.imread(ruta_img, cv2.IMREAD_GRAYSCALE)
    if img is None: return None
    region = img[y:y+h, x:x+w]
    if region.size == 0: return None
    var_region = float(np.var(region.astype(np.float32)))
    # Residualizar respecto a densidad de bordes global
    gx = cv2.Sobel(img, cv2.CV_64F, 1, 0, ksize=3)
    gy = cv2.Sobel(img, cv2.CV_64F, 0, 1, ksize=3)
    bordes = float(np.mean(np.sqrt(gx**2 + gy**2)))
    return var_region - reg_global * bordes

def generar_pares_imagen(imagen_id, dataset, ruta_izq, ruta_disp,
                          img_H, img_W, n_filas=3, n_cols=4):
    """Genera todos los pares A/B posibles dividiendo la imagen en cuadrícula."""
    h_celda = img_H // n_filas
    w_celda = img_W // n_cols

    # Calcular métrica para cada celda
    celdas = []
    reg_global = None

    if dataset == '3D_Illusion':
        # Calcular coeficiente de regresión global para residualización
        img = cv2.imread(ruta_izq, cv2.IMREAD_GRAYSCALE)
        if img is None: return []
        vars_img, bordes_img = [], []
        for fi in range(n_filas):
            for ci in range(n_cols):
                x = ci * w_celda
                y = fi * h_celda
                r = img[y:y+h_celda, x:x+w_celda]
                if r.size == 0: continue
                gx = cv2.Sobel(r.astype(np.float32), cv2.CV_64F, 1, 0)
                gy = cv2.Sobel(r.astype(np.float32), cv2.CV_64F, 0, 1)
                vars_img.append(float(np.var(r.astype(np.float32))))
                bordes_img.append(float(np.mean(np.sqrt(gx**2 + gy**2))))
        if len(vars_img) > 1:
            X = np.array(bordes_img).reshape(-1,1)
            y_arr = np.array(vars_img)
            reg = LinearRegression().fit(X, y_arr)
            reg_global = reg.coef_[0]

    for fi in range(n_filas):
        for ci in range(n_cols):
            x = ci * w_celda
            y = fi * h_celda
            if dataset == 'KITTI' and ruta_disp and os.path.exists(str(ruta_disp)):
                metrica = calcular_metrica_region_kitti(
                    str(ruta_disp), x, y, w_celda, h_celda)
            elif dataset == '3D_Illusion' and reg_global is not None:
                metrica = calcular_metrica_region_ilusion(
                    str(ruta_izq), x, y, w_celda, h_celda, reg_global)
            else:
                metrica = None
            if metrica is not None:
                celdas.append({'x': x, 'y': y, 'w': w_celda, 'h': h_celda,
                               'metrica': metrica})

    # Generar todos los pares con diferencia significativa
    pares = []
    metricas = [c['metrica'] for c in celdas]
    umbral_diff = np.std(metricas) * 0.5 if len(metricas) > 1 else 0

    for i in range(len(celdas)):
        for j in range(len(celdas)):
            if i == j: continue
            diff = celdas[i]['metrica'] - celdas[j]['metrica']
            if abs(diff) < umbral_diff: continue  # descartar pares ambiguos
            gt = 'mas_cercano' if diff > 0 else 'mas_lejano'
            pares.append({
                'imagen_id':   imagen_id,
                'dataset':     dataset,
                'ruta_img_izq': ruta_izq,
                'ruta_disp':   ruta_disp,
                'A_x': celdas[i]['x'], 'A_y': celdas[i]['y'],
                'A_ancho': celdas[i]['w'], 'A_alto': celdas[i]['h'],
                'A_metrica': celdas[i]['metrica'],
                'B_x': celdas[j]['x'], 'B_y': celdas[j]['y'],
                'B_ancho': celdas[j]['w'], 'B_alto': celdas[j]['h'],
                'B_metrica': celdas[j]['metrica'],
                'ground_truth': gt,
                'img_H': img_H, 'img_W': img_W
            })
    return pares


# Cargar pares base
df_pairs = pd.read_csv(os.path.join(SPLITS_DIR, 'pairs_ground_truth.csv'))
print(f'Pares base: {len(df_pairs)}')
print('Generando pares aumentados...')

todos_pares = []
errores = 0

for idx, row in df_pairs.iterrows():
    try:
        pares = generar_pares_imagen(
            imagen_id  = row['imagen_id'],
            dataset    = row['dataset'],
            ruta_izq   = str(row['ruta_img_izq']),
            ruta_disp  = str(row.get('ruta_disp', '')),
            img_H      = int(row['img_H']),
            img_W      = int(row['img_W'])
        )
        todos_pares.extend(pares)
    except Exception as e:
        errores += 1

    if (idx + 1) % 40 == 0:
        print(f'  {idx+1}/200 imágenes procesadas | {len(todos_pares)} pares generados')

df_aumentado = pd.DataFrame(todos_pares)
print(f'\nTotal pares generados: {len(df_aumentado)}')
print(f'Errores: {errores}')
print()
print('Por dataset:')
print(df_aumentado['dataset'].value_counts().to_string())
print()
print('Balance ground truth:')
print(df_aumentado['ground_truth'].value_counts().to_string())

# Guardar dataset aumentado
ruta_aumentado = os.path.join(SPLITS_DIR, 'pairs_train_augmented.csv')
df_aumentado.to_csv(ruta_aumentado, index=False)
print(f'\nGuardado en: {ruta_aumentado}')

Pares base: 200
Generando pares aumentados...
  40/200 imágenes procesadas | 1768 pares generados
  80/200 imágenes procesadas | 3580 pares generados
  120/200 imágenes procesadas | 6456 pares generados
  160/200 imágenes procesadas | 10380 pares generados
  200/200 imágenes procesadas | 14370 pares generados

Total pares generados: 14370
Errores: 0

Por dataset:
dataset
3D_Illusion    9898
KITTI          4472

Balance ground truth:
ground_truth
mas_cercano    7185
mas_lejano     7185

Guardado en: /content/drive/MyDrive/cognitive-depth-model/data/splits/pairs_train_augmented.csv


## Celda 3 — Dataset de pares para entrenamiento

In [3]:
class PairsDataset(Dataset):
    """
    Dataset de pares A/B para entrenamiento con ROI comparativo.
    Entrada: imagen completa (6 canales) + coordenadas A y B normalizadas.
    Etiqueta: 1.0 si A es más cercano que B, 0.0 si es más lejano.
    """
    def __init__(self, df, target_size=(192, 640), augment=False):
        self.df          = df.reset_index(drop=True)
        self.target_size = target_size
        self.augment     = augment
        self.mean = np.array([0.485,0.456,0.406,0.485,0.456,0.406], dtype=np.float32)
        self.std  = np.array([0.229,0.224,0.225,0.229,0.224,0.225], dtype=np.float32)

        # Filtrar filas con imágenes existentes
        validas = []
        for i, row in self.df.iterrows():
            if os.path.exists(str(row['ruta_img_izq'])):
                validas.append(i)
        self.df = self.df.loc[validas].reset_index(drop=True)
        print(f'PairsDataset: {len(self.df)} pares válidos')

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        h_target, w_target = self.target_size

        # Cargar imagen izquierda
        img_izq = cv2.imread(str(row['ruta_img_izq']))
        if img_izq is None:
            img_izq = np.zeros((375, 1242, 3), dtype=np.uint8)

        img_H_orig = row['img_H']
        img_W_orig = row['img_W']

        # Para KITTI cargar imagen derecha, para ilusiones duplicar
        if row['dataset'] == 'KITTI':
            ruta_der = str(row['ruta_img_izq']).replace('/image_2/', '/image_3/')
            img_der  = cv2.imread(ruta_der) if os.path.exists(ruta_der) else img_izq
            if img_der is None: img_der = img_izq
        else:
            img_der = img_izq.copy()

        img_izq = cv2.resize(cv2.cvtColor(img_izq, cv2.COLOR_BGR2RGB), (w_target, h_target))
        img_der = cv2.resize(cv2.cvtColor(img_der, cv2.COLOR_BGR2RGB), (w_target, h_target))

        # Escalar coordenadas al nuevo tamaño
        sx = w_target / img_W_orig
        sy = h_target / img_H_orig

        def escalar_roi(x, y, w, h):
            x2 = min(int(x * sx), w_target - 1)
            y2 = min(int(y * sy), h_target - 1)
            w2 = max(int(w * sx), 1)
            h2 = max(int(h * sy), 1)
            # Normalizar a [0,1] relativo al tamaño de la imagen
            return np.array([
                x2 / w_target, y2 / h_target,
                (x2 + w2) / w_target, (y2 + h2) / h_target
            ], dtype=np.float32)

        roi_A = escalar_roi(row['A_x'], row['A_y'], row['A_ancho'], row['A_alto'])
        roi_B = escalar_roi(row['B_x'], row['B_y'], row['B_ancho'], row['B_alto'])

        label = 1.0 if row['ground_truth'] == 'mas_cercano' else 0.0

        # Augmentación: flip horizontal con inversión de ROIs
        if self.augment and np.random.rand() > 0.5:
            img_izq = np.fliplr(img_izq).copy()
            img_der = np.fliplr(img_der).copy()
            # Invertir coordenadas x
            roi_A[0], roi_A[2] = 1 - roi_A[2], 1 - roi_A[0]
            roi_B[0], roi_B[2] = 1 - roi_B[2], 1 - roi_B[0]

        # Normalizar imagen
        par = np.concatenate([img_izq, img_der], axis=2).astype(np.float32) / 255.0
        par = (par - self.mean) / self.std
        img_tensor = torch.from_numpy(par.transpose(2, 0, 1))

        return (
            img_tensor,
            torch.from_numpy(roi_A),
            torch.from_numpy(roi_B),
            torch.tensor(label, dtype=torch.float32)
        )


# Cargar dataset aumentado y hacer split 80/20
df_aumentado = pd.read_csv(os.path.join(SPLITS_DIR, 'pairs_train_augmented.csv'))

# Shuffle y split
df_aumentado = df_aumentado.sample(frac=1, random_state=SEED).reset_index(drop=True)
n_train = int(len(df_aumentado) * 0.8)
df_train = df_aumentado.iloc[:n_train]
df_test  = df_aumentado.iloc[n_train:]

print(f'Split: {len(df_train)} train | {len(df_test)} test')
print()

TARGET_SIZE = (128, 384)
BATCH_SIZE  = 8

ds_train = PairsDataset(df_train, target_size=TARGET_SIZE, augment=True)
ds_test  = PairsDataset(df_test,  target_size=TARGET_SIZE, augment=False)

train_loader = DataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
test_loader  = DataLoader(ds_test,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f'Batches por época: {len(train_loader)}')

# Verificar un batch
img, roi_a, roi_b, lbl = next(iter(train_loader))
print(f'Batch: img={img.shape} roi_A={roi_a.shape} roi_B={roi_b.shape} labels={lbl.tolist()}')

Split: 11496 train | 2874 test

PairsDataset: 11496 pares válidos
PairsDataset: 2874 pares válidos
Batches por época: 1437
Batch: img=torch.Size([8, 6, 128, 384]) roi_A=torch.Size([8, 4]) roi_B=torch.Size([8, 4]) labels=[0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 1.0]


## Celda 4 — Arquitectura del modelo con ROI Pooling

In [4]:
class ResBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(ch, ch, 3, padding=1, bias=False), nn.BatchNorm2d(ch), nn.ReLU(inplace=True),
            nn.Conv2d(ch, ch, 3, padding=1, bias=False), nn.BatchNorm2d(ch))
        self.relu = nn.ReLU(inplace=True)
    def forward(self, x): return self.relu(x + self.net(x))

class ConvBnSkip(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.main = nn.Sequential(nn.Conv2d(in_ch,out_ch,3,padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))
        self.skip = nn.Sequential(nn.Conv2d(in_ch,out_ch,1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))
    def forward(self, x): return self.main(x) + self.skip(x)

class VisualBackbone(nn.Module):
    """
    Backbone bioinspirado: NGL → V1 → V2 → V3 → V4 → V5/MT
    Produce un mapa de features F de dimensión (B, 128, H/8, W/8)
    que captura información de profundidad a múltiples escalas.
    """
    def __init__(self):
        super().__init__()
        # NGL — Núcleo Geniculado Lateral
        self.ngl_magno = nn.Sequential(
            nn.Conv2d(6,8,5,padding=2), nn.BatchNorm2d(8), nn.ReLU(inplace=True), ResBlock(8))
        self.ngl_parvo = nn.Sequential(
            nn.Conv2d(6,8,3,padding=1), nn.BatchNorm2d(8), nn.ReLU(inplace=True), ResBlock(8))
        # V1 — Corteza visual primaria
        self.v1_merge = nn.Sequential(
            nn.Conv2d(16,16,1), nn.BatchNorm2d(16), nn.ReLU(inplace=True), ResBlock(16), ResBlock(16))
        self.pool1 = nn.MaxPool2d(2)  # /2
        # V2
        self.v2_thick = nn.Sequential(ConvBnSkip(16,32), ResBlock(32))
        self.v2_thin  = nn.Sequential(ConvBnSkip(16,32), ResBlock(32))
        self.v2_merge = nn.Sequential(nn.Conv2d(64,32,1), nn.BatchNorm2d(32), nn.ReLU(inplace=True))
        self.pool2 = nn.MaxPool2d(2)  # /4
        # V3
        self.v3_disp   = nn.Sequential(ConvBnSkip(32,64), ResBlock(64), ResBlock(64))
        self.v3_motion = nn.Sequential(ConvBnSkip(32,64), ResBlock(64))
        self.pool3 = nn.MaxPool2d(2)  # /8
        # V4
        self.v4 = nn.Sequential(
            nn.Conv2d(128,64,1), nn.BatchNorm2d(64), nn.ReLU(inplace=True), ResBlock(64))
        # V5/MT — movimiento y disparidad
        self.v5mt = nn.Sequential(
            nn.Conv2d(64,128,3,padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            ResBlock(128), ResBlock(128))
        # Feedback V2→V1
        self.fb_v2_v1 = nn.Sequential(nn.Conv2d(32,16,1), nn.BatchNorm2d(16), nn.ReLU(inplace=True))

    def forward(self, x):
        # NGL
        magno = self.ngl_magno(x)
        parvo = self.ngl_parvo(x)
        # V1 con feedback
        v1 = self.v1_merge(torch.cat([magno, parvo], dim=1))
        v1 = self.pool1(v1)
        # V2
        thick = self.v2_thick(v1)
        thin  = self.v2_thin(v1)
        v2    = self.v2_merge(torch.cat([thick, thin], dim=1))
        # Feedback: V2 → V1 (segundo pase)
        fb   = F.interpolate(self.fb_v2_v1(v2), size=v1.shape[2:], mode='bilinear', align_corners=False)
        v1   = v1 + fb
        thick = self.v2_thick(v1)
        thin  = self.v2_thin(v1)
        v2    = self.v2_merge(torch.cat([thick, thin], dim=1))
        v2    = self.pool2(v2)
        # V3
        disp   = self.v3_disp(v2)
        motion = self.v3_motion(v2)
        v3     = torch.cat([disp, motion], dim=1)
        v3     = self.pool3(v3)
        # V4 + V5/MT
        v4   = self.v4(v3)
        v5mt = self.v5mt(v4)
        return v5mt  # (B, 128, H/8, W/8)


class ROIComparator(nn.Module):
    """
    Extrae features de regiones A y B mediante ROI Pooling y las compara.
    Replica el mecanismo de atención selectiva: procesa la escena globalmente
    y luego focaliza en los objetos relevantes para la comparación de profundidad.
    """
    def __init__(self, feat_dim=128, roi_size=4):
        super().__init__()
        self.roi_size = roi_size
        feat_total = feat_dim * roi_size * roi_size
        self.comparator = nn.Sequential(
            # Concatenar features A, features B, y diferencia A-B
            nn.Linear(feat_total * 3, 512), nn.ReLU(inplace=True), nn.Dropout(0.3),
            nn.Linear(512, 128),           nn.ReLU(inplace=True), nn.Dropout(0.3),
            nn.Linear(128, 1),             nn.Sigmoid()
        )

    def roi_pool(self, features, roi):
        """
        ROI Pooling diferenciable usando grid_sample.
        roi: (B, 4) con valores normalizados [x1, y1, x2, y2] en [0,1]
        """
        B, C, H, W = features.shape
        # Convertir de [0,1] a [-1,1] para grid_sample
        x1 = roi[:, 0] * 2 - 1
        y1 = roi[:, 1] * 2 - 1
        x2 = roi[:, 2] * 2 - 1
        y2 = roi[:, 3] * 2 - 1

        # Crear grid para la región
        xs = torch.stack([x1 + (x2 - x1) * t for t in
                          torch.linspace(0, 1, self.roi_size).to(features.device)], dim=1)
        ys = torch.stack([y1 + (y2 - y1) * t for t in
                          torch.linspace(0, 1, self.roi_size).to(features.device)], dim=1)

        # Grid (B, roi_size, roi_size, 2)
        grid_x = xs.unsqueeze(1).expand(B, self.roi_size, self.roi_size)
        grid_y = ys.unsqueeze(2).expand(B, self.roi_size, self.roi_size)
        grid   = torch.stack([grid_x, grid_y], dim=-1)

        # Extraer features de la región
        roi_feat = F.grid_sample(features, grid, align_corners=True, mode='bilinear')
        return roi_feat.flatten(1)  # (B, C * roi_size * roi_size)

    def forward(self, features, roi_A, roi_B):
        feat_A = self.roi_pool(features, roi_A)
        feat_B = self.roi_pool(features, roi_B)
        diff   = feat_A - feat_B
        x      = torch.cat([feat_A, feat_B, diff], dim=1)
        return self.comparator(x)


class CognitiveDepthModelV2(nn.Module):
    """Modelo cognitivo con backbone bioinspirado + ROI comparativo."""
    def __init__(self):
        super().__init__()
        self.backbone = VisualBackbone()
        self.comparator = ROIComparator(feat_dim=128, roi_size=4)

    def forward(self, img, roi_A, roi_B):
        features = self.backbone(img)
        return self.comparator(features, roi_A, roi_B)


model = CognitiveDepthModelV2().to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f'✓ Modelo v2 cargado: {total_params:,} parámetros')

# Verificar forward pass
with torch.no_grad():
    img_t  = torch.randn(2, 6, 128, 384).to(device)
    roi_at = torch.tensor([[0.1, 0.1, 0.5, 0.5], [0.2, 0.2, 0.6, 0.6]]).to(device)
    roi_bt = torch.tensor([[0.5, 0.5, 0.9, 0.9], [0.6, 0.6, 0.9, 0.9]]).to(device)
    out    = model(img_t, roi_at, roi_bt)
    print(f'  Forward pass OK: {out.shape} | valores: {out.squeeze().tolist()}')

✓ Modelo v2 cargado: 4,287,185 parámetros
  Forward pass OK: torch.Size([2, 1]) | valores: [0.5749115347862244, 0.6105973720550537]


## Celda 5 — Funciones de entrenamiento

In [5]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, total_correct, total_n = 0.0, 0, 0
    all_preds, all_labels = [], []
    for img, roi_a, roi_b, y in loader:
        img, roi_a, roi_b, y = img.to(device), roi_a.to(device), roi_b.to(device), y.to(device)
        optimizer.zero_grad()
        out  = model(img, roi_a, roi_b).squeeze()
        loss = criterion(out, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        preds = (out > 0.5).float()
        total_loss    += loss.item() * len(y)
        total_correct += (preds == y).sum().item()
        total_n       += len(y)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.cpu().numpy())
    f1 = f1_score(all_labels, all_preds, zero_division=0)
    return {'loss': total_loss/total_n, 'accuracy': total_correct/total_n, 'f1': f1}


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, total_correct, total_n = 0.0, 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for img, roi_a, roi_b, y in loader:
            img, roi_a, roi_b, y = img.to(device), roi_a.to(device), roi_b.to(device), y.to(device)
            out  = model(img, roi_a, roi_b).squeeze()
            loss = criterion(out, y)
            preds = (out > 0.5).float()
            total_loss    += loss.item() * len(y)
            total_correct += (preds == y).sum().item()
            total_n       += len(y)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())
    f1 = f1_score(all_labels, all_preds, zero_division=0)
    return {'loss': total_loss/total_n, 'accuracy': total_correct/total_n, 'f1': f1}


class EarlyStopping:
    def __init__(self, patience=15, min_delta=0.001):
        self.patience  = patience
        self.min_delta = min_delta
        self.best      = float('inf')
        self.counter   = 0
        self.stop      = False
    def __call__(self, val_loss):
        if val_loss < self.best - self.min_delta:
            self.best    = val_loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience: self.stop = True


def freeze_backbone_early(model):
    """Congela NGL y V1 del backbone para Etapa 1."""
    for name, param in model.backbone.named_parameters():
        if name.startswith('ngl_') or name.startswith('v1_'):
            param.requires_grad = False
    frozen = sum(1 for p in model.parameters() if not p.requires_grad)
    print(f'  Parámetros congelados: {frozen}')

def unfreeze_all(model):
    for p in model.parameters(): p.requires_grad = True
    print(f'  Todos los parámetros descongelados.')


print('✓ Funciones de entrenamiento definidas.')

✓ Funciones de entrenamiento definidas.


## Celda 6 — Etapa 1: Entrenamiento con backbone parcialmente congelado (30 épocas)

In [14]:
import torch, gc
torch.cuda.empty_cache()
gc.collect()

model = CognitiveDepthModelV2().to(device)
freeze_backbone_early(model)

criterion  = nn.BCELoss()
optimizer  = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=0.001, weight_decay=1e-4)
scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, patience=10, factor=0.5, min_lr=1e-7)
early_stop = EarlyStopping(patience=15)

STAGE1_EPOCHS = 30
history_s1    = {'train': [], 'test': [], 'epoch': [], 'lr': []}
best_s1       = float('inf')

print('='*60)
print('ETAPA 1: Backbone parcialmente congelado (NGL + V1)')
print(f'Épocas: {STAGE1_EPOCHS} | lr: 0.001 | batch: {BATCH_SIZE}')
print('='*60)

for epoch in range(STAGE1_EPOCHS):
    t0      = time.time()
    train_m = train_one_epoch(model, train_loader, criterion, optimizer, device)
    test_m  = evaluate(model, test_loader, criterion, device)
    lr      = optimizer.param_groups[0]['lr']
    scheduler.step(test_m['loss'])
    elapsed = time.time() - t0

    history_s1['train'].append(train_m)
    history_s1['test'].append(test_m)
    history_s1['epoch'].append(epoch + 1)
    history_s1['lr'].append(lr)

    print(f'E{epoch+1:02d}/{STAGE1_EPOCHS} | '
          f'Train loss={train_m["loss"]:.4f} acc={train_m["accuracy"]:.3f} f1={train_m["f1"]:.3f} | '
          f'Test  loss={test_m["loss"]:.4f} acc={test_m["accuracy"]:.3f} | '
          f'lr={lr:.2e} | {elapsed:.0f}s')

    if test_m['loss'] < best_s1:
        best_s1 = test_m['loss']
        torch.save({
            'epoch': epoch+1, 'stage': 1,
            'model_state_dict': model.state_dict(),
            'test_loss': test_m['loss'], 'metrics': test_m
        }, os.path.join(CHECKPOINT_DIR, 'best_model_v2.pth'))

    early_stop(test_m['loss'])
    if early_stop.stop:
        print(f'  Early stopping en época {epoch+1}.')
        break

print(f'\n✓ Etapa 1 completada. Mejor test loss: {best_s1:.4f}')

  Parámetros congelados: 36
ETAPA 1: Backbone parcialmente congelado (NGL + V1)
Épocas: 30 | lr: 0.001 | batch: 8
E01/30 | Train loss=0.4024 acc=0.890 f1=0.890 | Test  loss=0.3720 acc=0.947 | lr=1.00e-03 | 439s
E02/30 | Train loss=0.2644 acc=0.945 f1=0.945 | Test  loss=0.1553 acc=0.960 | lr=1.00e-03 | 410s
E03/30 | Train loss=0.1932 acc=0.956 f1=0.956 | Test  loss=0.1047 acc=0.969 | lr=1.00e-03 | 405s
E04/30 | Train loss=0.1567 acc=0.963 f1=0.963 | Test  loss=0.1224 acc=0.966 | lr=1.00e-03 | 407s
E05/30 | Train loss=0.1277 acc=0.967 f1=0.967 | Test  loss=0.0896 acc=0.970 | lr=1.00e-03 | 409s
E06/30 | Train loss=0.1154 acc=0.972 f1=0.972 | Test  loss=0.1133 acc=0.975 | lr=1.00e-03 | 411s
E07/30 | Train loss=0.1214 acc=0.973 f1=0.973 | Test  loss=0.1297 acc=0.969 | lr=1.00e-03 | 414s
E08/30 | Train loss=0.0989 acc=0.975 f1=0.975 | Test  loss=0.0693 acc=0.976 | lr=1.00e-03 | 413s
E09/30 | Train loss=0.0981 acc=0.975 f1=0.975 | Test  loss=0.0719 acc=0.982 | lr=1.00e-03 | 408s
E10/30 | Trai

## Celda 7 — Etapa 2: Fine-tuning completo (70 épocas)

In [ ]:
ckpt = torch.load(os.path.join(CHECKPOINT_DIR, 'best_model_v2.pth'),
                  map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
print(f'Mejor Etapa 1 cargado (época {ckpt["epoch"]}, loss={ckpt["test_loss"]:.4f})')

unfreeze_all(model)

criterion  = nn.BCELoss()
optimizer  = torch.optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-4)
scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, patience=10, factor=0.5, min_lr=1e-7)
early_stop = EarlyStopping(patience=15)

STAGE2_EPOCHS = 70
history_s2    = {'train': [], 'test': [], 'epoch': [], 'lr': []}
best_s2       = ckpt['test_loss']

print()
print('='*60)
print('ETAPA 2: Fine-tuning completo')
print(f'Épocas: {STAGE2_EPOCHS} | lr: 0.0001 | batch: {BATCH_SIZE}')
print('='*60)

for epoch in range(STAGE2_EPOCHS):
    t0      = time.time()
    train_m = train_one_epoch(model, train_loader, criterion, optimizer, device)
    test_m  = evaluate(model, test_loader, criterion, device)
    lr      = optimizer.param_groups[0]['lr']
    scheduler.step(test_m['loss'])
    elapsed = time.time() - t0

    history_s2['train'].append(train_m)
    history_s2['test'].append(test_m)
    history_s2['epoch'].append(epoch + 1)
    history_s2['lr'].append(lr)

    print(f'E{epoch+1:02d}/{STAGE2_EPOCHS} | '
          f'Train loss={train_m["loss"]:.4f} acc={train_m["accuracy"]:.3f} f1={train_m["f1"]:.3f} | '
          f'Test  loss={test_m["loss"]:.4f} acc={test_m["accuracy"]:.3f} | '
          f'lr={lr:.2e} | {elapsed:.0f}s')

    if test_m['loss'] < best_s2:
        best_s2 = test_m['loss']
        torch.save({
            'epoch': epoch+1, 'stage': 2,
            'model_state_dict': model.state_dict(),
            'test_loss': test_m['loss'], 'metrics': test_m
        }, os.path.join(CHECKPOINT_DIR, 'best_model_v2.pth'))
        print(f'  ✓ Nuevo mejor modelo (loss={best_s2:.4f})')

    early_stop(test_m['loss'])
    if early_stop.stop:
        print(f'  Early stopping en época {epoch+1}.')
        break

torch.save({
    'epoch': epoch+1, 'stage': 2,
    'model_state_dict': model.state_dict(),
    'test_loss': test_m['loss'], 'metrics': test_m
}, os.path.join(CHECKPOINT_DIR, 'model_final_v2.pth'))

print(f'\n✓ Etapa 2 completada. Mejor test loss: {best_s2:.4f}')

Mejor Etapa 1 cargado (época 6, loss=0.0093)
  Todos los parámetros descongelados.

ETAPA 2: Fine-tuning completo
Épocas: 70 | lr: 0.0001 | batch: 8
E01/70 | Train loss=0.0245 acc=0.995 f1=0.995 | Test  loss=0.0451 acc=0.992 | lr=1.00e-04 | 888s
E02/70 | Train loss=0.0224 acc=0.994 f1=0.994 | Test  loss=0.0285 acc=0.993 | lr=1.00e-04 | 837s
E03/70 | Train loss=0.0250 acc=0.994 f1=0.994 | Test  loss=0.0435 acc=0.990 | lr=1.00e-04 | 830s
E04/70 | Train loss=0.0280 acc=0.995 f1=0.995 | Test  loss=0.0263 acc=0.994 | lr=1.00e-04 | 835s
E05/70 | Train loss=0.0222 acc=0.995 f1=0.995 | Test  loss=0.0263 acc=0.993 | lr=1.00e-04 | 843s
E06/70 | Train loss=0.0198 acc=0.996 f1=0.996 | Test  loss=0.0243 acc=0.994 | lr=1.00e-04 | 825s
E07/70 | Train loss=0.0209 acc=0.997 f1=0.997 | Test  loss=0.0344 acc=0.993 | lr=1.00e-04 | 821s
E08/70 | Train loss=0.0226 acc=0.996 f1=0.996 | Test  loss=0.0272 acc=0.994 | lr=1.00e-04 | 839s
E09/70 | Train loss=0.0163 acc=0.997 f1=0.997 | Test  loss=0.0148 acc=0.997

## Celda 8 — Visualización y evaluación final

In [7]:
import os, pandas as pd
import torch, torch.nn as nn
import matplotlib.pyplot as plt

BASE_DIR       = '/content/drive/MyDrive/cognitive-depth-model'
CHECKPOINT_DIR = os.path.join(BASE_DIR, 'checkpoints')
RESULTS_DIR    = os.path.join(BASE_DIR, 'results')
SPLITS_DIR     = os.path.join(BASE_DIR, 'data/splits')
TARGET_SIZE    = (128, 384)
BATCH_SIZE     = 8
device         = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Cargar mejor modelo
ckpt = torch.load(os.path.join(CHECKPOINT_DIR, 'best_model_v2.pth'),
                  map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

print('='*60)
print('EVALUACIÓN FINAL — MODELO v2 (KITTI + Ilusiones, ROI Pooling)')
print('='*60)
print(f'Mejor época: {ckpt["epoch"]} | Etapa: {ckpt["stage"]}')
print(f'Test loss:   {ckpt["test_loss"]:.4f}')
print(f'Test acc:    {ckpt["metrics"]["accuracy"]:.4f} ({ckpt["metrics"]["accuracy"]*100:.1f}%)')
print(f'Test F1:     {ckpt["metrics"]["f1"]:.4f}')
print()

# Evaluación separada por dataset
df_aug  = pd.read_csv(os.path.join(SPLITS_DIR, 'pairs_train_augmented.csv'))
df_aug  = df_aug.sample(frac=1, random_state=42).reset_index(drop=True)
df_test = df_aug.iloc[int(len(df_aug)*0.8):]

df_test_k = df_test[df_test['dataset']=='KITTI']
df_test_i = df_test[df_test['dataset']=='3D_Illusion']

ds_test_k = PairsDataset(df_test_k, target_size=TARGET_SIZE)
ds_test_i = PairsDataset(df_test_i, target_size=TARGET_SIZE)

loader_k = torch.utils.data.DataLoader(ds_test_k, batch_size=BATCH_SIZE, shuffle=False)
loader_i = torch.utils.data.DataLoader(ds_test_i, batch_size=BATCH_SIZE, shuffle=False)

criterion = nn.BCELoss()

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, total_correct, total_n = 0.0, 0, 0
    from sklearn.metrics import f1_score
    all_preds, all_labels = [], []
    with torch.no_grad():
        for img, roi_a, roi_b, y in loader:
            img, roi_a, roi_b, y = img.to(device), roi_a.to(device), roi_b.to(device), y.to(device)
            out   = model(img, roi_a, roi_b).squeeze()
            loss  = criterion(out, y)
            preds = (out > 0.5).float()
            total_loss    += loss.item() * len(y)
            total_correct += (preds == y).sum().item()
            total_n       += len(y)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())
    f1 = f1_score(all_labels, all_preds, zero_division=0)
    return {'loss': total_loss/total_n, 'accuracy': total_correct/total_n, 'f1': f1}

m_k = evaluate(model, loader_k, criterion, device)
m_i = evaluate(model, loader_i, criterion, device)

print('Desempeño por dataset:')
print(f'  KITTI:      acc={m_k["accuracy"]:.4f} ({m_k["accuracy"]*100:.1f}%) | f1={m_k["f1"]:.4f}')
print(f'  Ilusiones:  acc={m_i["accuracy"]:.4f} ({m_i["accuracy"]*100:.1f}%) | f1={m_i["f1"]:.4f}')

# Guardar resumen
resumen = pd.DataFrame([{
    'modelo': 'v2_ROI_KITTI_Ilusiones',
    'mejor_epoca': ckpt['epoch'],
    'test_loss': ckpt['test_loss'],
    'test_accuracy': ckpt['metrics']['accuracy'],
    'test_f1': ckpt['metrics']['f1'],
    'acc_kitti': m_k['accuracy'],
    'acc_ilusiones': m_i['accuracy'],
}])
ruta = os.path.join(RESULTS_DIR, 'metrics', 'model_v2_final_metrics.csv')
os.makedirs(os.path.dirname(ruta), exist_ok=True)
resumen.to_csv(ruta, index=False)
print(f'\n✓ Métricas guardadas en {ruta}')

EVALUACIÓN FINAL — MODELO v2 (KITTI + Ilusiones, ROI Pooling)
Mejor época: 25 | Etapa: 2
Test loss:   0.0030
Test acc:    0.9986 (99.9%)
Test F1:     0.9986

PairsDataset: 869 pares válidos
PairsDataset: 2005 pares válidos
Desempeño por dataset:
  KITTI:      acc=0.9988 (99.9%) | f1=0.9989
  Ilusiones:  acc=0.9985 (99.9%) | f1=0.9985

✓ Métricas guardadas en /content/drive/MyDrive/cognitive-depth-model/results/metrics/model_v2_final_metrics.csv
